# 6 · Anotações próprias e avaliação conjunta

A proposta exige fotografias de resistores separados com caixas, cores, valor nominal e tolerância. Este notebook valida as anotações e cria a divisão sem misturar cenas ou identidades conhecidas. Nenhuma anotação é inventada quando as fotos ou os rótulos não estão disponíveis.

Crie `data/inputs/proprias/anotacoes.jsonl` (ou defina `RESISTOR_OWN_ANNOTATIONS`). Cada linha descreve uma imagem; caixas são `[x1,y1,x2,y2]` em pixels. As faixas estão na ordem de leitura. Exemplo de **formato**, não um dado real:

```json
{"id":"foto-001","image":"foto-001.jpg","scene_id":"cena-01","objects":[{"component_id":"peca-01","bbox":[100,50,240,105],"bands":["yellow","violet","brown","gold"],"ohms":470,"tolerance_percent":5,"orientation":"esquerda-direita"}]}
```

Identifique a mesma peça com o mesmo `component_id` em todas as fotos e a mesma cena com o mesmo `scene_id`. Para leituras não verificáveis use `null` nos respectivos campos. Para medir valor e tolerância, use rótulos conferidos visualmente ou por medição independente.

In [ ]:
from pathlib import Path
import os,sys,json,random
from collections import defaultdict,Counter
import cv2,numpy as np,pandas as pd
def read_image(path):
    try: return cv2.imdecode(np.fromfile(str(path),dtype=np.uint8),cv2.IMREAD_COLOR)
    except (OSError,cv2.error): return None
PROJECT=Path(os.environ.get("RESISTOR_PROJECT_ROOT","..")).expanduser().resolve()
DATA=Path(os.environ.get("RESISTOR_DATA_ROOT",str(PROJECT/"data"))).expanduser().resolve()
OUTPUT=Path(os.environ.get("RESISTOR_OUTPUT",str(PROJECT/"outputs/proposta"))).expanduser().resolve()
ANNOTATIONS=Path(os.environ.get("RESISTOR_OWN_ANNOTATIONS",str(DATA/"inputs/proprias/anotacoes.jsonl"))).expanduser().resolve()
sys.path.insert(0,str(PROJECT/"src"))
from resistor_detector.colors import decode_color_bands
rows=[]
if ANNOTATIONS.exists():
    rows=[json.loads(s) for s in ANNOTATIONS.read_text(encoding="utf-8").splitlines() if s.strip()]
else:
    print("Ainda não há anotações próprias em", ANNOTATIONS)
print("Imagens anotadas:",len(rows))


## Validação das anotações

In [ ]:
def validate_annotations(rows):
    errors=[]; ids=set()
    for record in rows:
        ident=record.get("id")
        if not ident or ident in ids:errors.append(f"{ident}: id ausente ou repetido")
        ids.add(ident)
        if not record.get("scene_id"):errors.append(f"{ident}: scene_id ausente")
        image_path=(ANNOTATIONS.parent/record.get("image","")).resolve()
        if not image_path.is_relative_to(ANNOTATIONS.parent):
            errors.append(f"{ident}: caminho fora da pasta");continue
        image=read_image(image_path)
        if image is None: errors.append(f"{ident}: imagem ausente {image_path}");continue
        h,w=image.shape[:2]
        if not isinstance(record.get("objects"),list):errors.append(f"{ident}: objects deve ser lista");continue
        components=set()
        for obj in record["objects"]:
            component=obj.get("component_id")
            if not component or component in components:errors.append(f"{ident}: component_id ausente ou repetido na imagem")
            components.add(component)
            box=obj.get("bbox",[])
            if len(box)!=4 or not all(isinstance(x,(int,float)) for x in box):
                errors.append(f"{ident}/{component}: caixa inválida");continue
            x1,y1,x2,y2=box
            if not (0<=x1<x2<=w and 0<=y1<y2<=h):errors.append(f"{ident}/{component}: caixa fora da imagem")
            bands=obj.get("bands")
            if bands is not None:
                value=decode_color_bands(bands) if len(bands) in (4,5) and all(isinstance(c,str) for c in bands) else None
                if value is None or len(value.bands)!=len(bands):errors.append(f"{ident}/{component}: faixas inválidas")
                elif obj.get("ohms") is not None and not np.isclose(value.ohms,obj["ohms"]):errors.append(f"{ident}/{component}: valor diverge das faixas")
                elif obj.get("tolerance_percent") is not None and value.tolerance_percent!=obj["tolerance_percent"]:errors.append(f"{ident}/{component}: tolerância diverge das faixas")
    return errors
errors=validate_annotations(rows)
if errors:
    for error in errors[:30]:print(error)
    raise ValueError(f"{len(errors)} erro(s) de anotação.")
if rows:print("Anotações verificadas:",len(rows),"imagens;",sum(len(r["objects"]) for r in rows),"resistores.")


## Divisão por cena e componente

Grupos conectados por qualquer cena ou identidade física ficam na mesma partição. Com poucos grupos não é possível garantir 70/15/15 ou conclusões estatísticas confiáveis.

In [ ]:
if rows:
    parent=list(range(len(rows)))
    def find(i):
        while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]
        return i
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b:parent[b]=a
    seen={}
    for i,row in enumerate(rows):
        keys=[("scene",row["scene_id"])]+[("component",o["component_id"]) for o in row["objects"]]
        for key in keys:
            if key in seen:union(i,seen[key])
            else:seen[key]=i
    grouped=defaultdict(list)
    for i in range(len(rows)):grouped[find(i)].append(i)
    if len(grouped)<3:
        print("São necessários pelo menos três grupos independentes para separar treino, validação e teste.")
    else:
        groups=list(grouped.values());random.Random(42).shuffle(groups);groups.sort(key=len,reverse=True)
        targets={"train":.7*len(rows),"val":.15*len(rows),"test":.15*len(rows)}
        assigned={k:[] for k in targets}
        for i,group in enumerate(groups):
            # Garanta que cada partição tenha ao menos um grupo.
            destination=list(targets)[i] if i<3 else max(targets,key=lambda k:(targets[k]-len(assigned[k]))/targets[k])
            assigned[destination].extend(group)
        OUTPUT.mkdir(parents=True,exist_ok=True)
        split_rows=[{**rows[index],"split":split} for split,indexes in assigned.items() for index in indexes]
        output=OUTPUT/"proprias_divisao.jsonl"
        output.write_text("".join(json.dumps(r,ensure_ascii=False)+"\n" for r in split_rows),encoding="utf-8")
        print("Grupos:",len(groups),"imagens:",{k:len(v) for k,v in assigned.items()},"\nManifesto:",output)


## Comparação de previsões com o conjunto anotado

Previsões podem ser exportadas como JSONL, uma linha por imagem: `{"id":"foto-001","objects":[{"bbox":[...],"ohms":470,"tolerance_percent":5}]}`. Informe o caminho em `RESISTOR_OWN_PREDICTIONS`. A avaliação faz associação um a um por IoU ≥ 0,5. Objetos não detectados ou sem leitura contam como erros; rótulos desconhecidos são excluídos apenas das métricas para as quais faltam anotações.

In [ ]:
def overlap(a,b):
    inter=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    return inter/max(1e-9,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter)
def evaluate_image(truth,pred):
    candidates=sorted(((overlap(a["bbox"],b["bbox"]),i,j) for i,a in enumerate(truth) for j,b in enumerate(pred)),reverse=True)
    a_used,b_used=set(),set();pairs=[]
    for iou,i,j in candidates:
        if iou<.5:break
        if i not in a_used and j not in b_used:a_used.add(i);b_used.add(j);pairs.append((i,j))
    return pairs,len(pred)-len(b_used),len(truth)-len(a_used)
assert evaluate_image([{"bbox":[0,0,10,10]}],[{"bbox":[0,0,10,10]},{"bbox":[0,0,10,10]}])[1:]==(1,0)
pred_path=os.environ.get("RESISTOR_OWN_PREDICTIONS")
if rows and pred_path:
    predictions=[json.loads(s) for s in Path(pred_path).read_text(encoding="utf-8").splitlines() if s.strip()]
    by_id={r["id"]:r for r in predictions}
    if len(by_id)!=len(predictions):raise ValueError("Ids duplicados nas previsões")
    split_path=OUTPUT/"proprias_divisao.jsonl"
    if not split_path.exists():raise FileNotFoundError("Crie primeiro uma divisão válida.")
    test=[json.loads(s) for s in split_path.read_text(encoding="utf-8").splitlines() if json.loads(s)["split"]=="test"]
    counts=[];latencies=[];tp=fp=fn=0;value_total=value_ok=tol_total=tol_ok=within_total=within_ok=0
    for row in test:
        truth=row["objects"];pred=by_id.get(row["id"],{}).get("objects",[])
        latency=by_id.get(row["id"],{}).get("latency_ms")
        if latency is not None:latencies.append(float(latency))
        pairs,fpos,fneg=evaluate_image(truth,pred);tp+=len(pairs);fp+=fpos;fn+=fneg;counts.append(abs(len(truth)-len(pred)))
        paired=dict(pairs)
        for i,obj in enumerate(truth):
            guess=pred[paired[i]] if i in paired else {}
            if obj.get("ohms") is not None:
                value_total+=1;v=guess.get("ohms");value_ok+=int(v is not None and np.isclose(v,obj["ohms"]))
            if obj.get("tolerance_percent") is not None:
                tol_total+=1;t=guess.get("tolerance_percent");tol_ok+=int(t is not None and np.isclose(t,obj["tolerance_percent"]))
            if obj.get("ohms") is not None and obj.get("tolerance_percent") is not None:
                within_total+=1;v=guess.get("ohms");within_ok+=int(v is not None and abs(v-obj["ohms"])<=abs(obj["ohms"])*obj["tolerance_percent"]/100)
    report={"imagens":len(test),"contagem_exata":sum(x==0 for x in counts)/len(counts) if counts else None,"erro_absoluto_medio":float(np.mean(counts)) if counts else None,
        "precisao_iou50":tp/(tp+fp) if tp+fp else None,"recall_iou50":tp/(tp+fn) if tp+fn else None,
        "valor_exato":value_ok/value_total if value_total else None,"total_valores_rotulados":value_total,
        "tolerancia_exata":tol_ok/tol_total if tol_total else None,"total_tolerancias_rotuladas":tol_total,
        "dentro_tolerancia":within_ok/within_total if within_total else None,"tempo_medio_ms":float(np.mean(latencies)) if latencies else None}
    display(pd.DataFrame([report]))
else:print("Avaliação conjunta aguardando anotações e arquivo de previsões.")
